In [2]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.metrics import classification_report

In [3]:
# ===================== Load Data =====================
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

In [4]:
# ===================== Preprocess =====================
def clean_title(title):
    return re.sub("[^a-zA-Z0-9 ]", "", title)

movies["clean_title"] = movies["title"].apply(clean_title)

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(movies["clean_title"])

In [5]:
# ===================== Search Function =====================
def search(title):
    title = clean_title(title)
    query_vec = vectorizer.transform([title])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]
    return results


In [6]:

# ===================== Improved find_similar_movies =====================
def find_similar_movies(movie_id):
    similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()

    similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]

    # Normalize by the number of similar users
    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

    # Lowered threshold to include more recommendations
    similar_user_recs = similar_user_recs[similar_user_recs > .02]

    all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]
    all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

    rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]

    # Refined score: favor similarity, reduce popularity bias
    rec_percentages["score"] = (rec_percentages["similar"] ** 1.5) / np.sqrt(rec_percentages["all"])

    rec_percentages = rec_percentages.sort_values("score", ascending=False)
    result = rec_percentages.head(20).merge(movies, left_index=True, right_on="movieId")

    return result[["movieId", "score", "title", "genres"]]


In [7]:
# ===================== Example Usage =====================
display(search("Toy"))

,movieId,title,genres,clean_title
3595,4929,"Toy, The (1982)",Comedy,Toy The 1982
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Toy Story 3 2010
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 2 1999
4089,5843,Toy Soldiers (1991),Action|Drama,Toy Soldiers 1991


In [8]:
display(find_similar_movies(1))

,movieId,score,title,genres
0,1,3.015345,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2355,3114,0.722290,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
277,318,0.483599,"Shawshank Redemption, The (1994)",Crime|Drama
510,593,0.482098,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
257,296,0.475911,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
314,356,0.465766,Forrest Gump (1994),Comedy|Drama|Romance|War
322,364,0.465177,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
506,588,0.437749,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
224,260,0.423109,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
7355,78499,0.420810,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX


In [9]:
# ===================== Train-Test Split =====================
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)
ratings = train_ratings


In [10]:
# ===================== Improved Evaluation Function =====================
def evaluate_model(k=10):
    all_precisions = []
    all_recalls = []
    all_f1s = []

    # Build the test data likes
    test_user_likes = defaultdict(set)
    for _, row in test_ratings.iterrows():
        if row["rating"] >= 4:
            test_user_likes[row["userId"]].add(row["movieId"])

    # Increase the number of test users for evaluation (optional)
    test_users = list(test_user_likes.keys())[:200]  # Try 200 users now

    for user in test_users:
        liked_movies = test_user_likes[user]

        # Get this user's liked movies from training data
        user_train_movies = train_ratings[(train_ratings["userId"] == user) & (train_ratings["rating"] >= 4)]["movieId"].tolist()

        if not user_train_movies or not liked_movies:
            continue  # Skip users with no train data or no liked movies in test

        # Use top 5 liked movies to improve recommendations
        all_recommended = pd.DataFrame()

        for query_movie in user_train_movies[:5]:  # Using up to 5 seed movies
            recommended = find_similar_movies(query_movie)
            all_recommended = pd.concat([all_recommended, recommended])

        # Aggregate scores from multiple queries
        all_recommended = all_recommended.groupby("movieId").agg({
            "score": "sum",
            "title": "first",
            "genres": "first"
        }).reset_index()

        all_recommended = all_recommended.sort_values("score", ascending=False)

        # Remove movies already rated by the user in training
        recommended_movies = [m for m in all_recommended["movieId"].tolist() if m not in user_train_movies][:k]

        # Calculate hits
        hits = len(set(recommended_movies) & liked_movies)

        # Calculate precision, recall, and f1
        precision = hits / k if k else 0
        recall = hits / len(liked_movies) if liked_movies else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0

        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)

    # Calculate the mean across all users
    avg_precision = np.mean(all_precisions)
    avg_recall = np.mean(all_recalls)
    avg_f1 = np.mean(all_f1s)

    print(f"Overall Precision@{k}: {avg_precision:.4f}")
    print(f"Overall Recall@{k}: {avg_recall:.4f}")
    print(f"Overall F1-Score@{k}: {avg_f1:.4f}")


In [11]:

# ===================== Run Evaluation =====================
evaluate_model(k=10)

Overall Precision@10: 0.0370
Overall Recall@10: 0.0258
Overall F1-Score@10: 0.0248
